# Universal API Components — talking to a real AI
### Plain-English edition — no coding required

Everything so far ran on your own machine. Now you will send a message over the
internet to a real language model and get a real answer back.

Every AI service works the same five ways, and after this you will recognise them
in any documentation you ever read.

## How this notebook works
Every grey box below is a **step you run**, not code you read.

1. Some steps have **boxes to fill in** (type a word, drag a slider, tick a box).
2. Press the **▶ play button** on the left of the step.
3. Read the answer that appears underneath.

That's it. The computer writes and runs the code for you.

**Curious what it wrote?** Click **Show code** on any step to read it, then click
**Hide code** in the same place to tuck it away again.

> **Red text means something needs fixing** — read it and press play again.
> **Green means you got it right.** No red at all? You're good.

## Before you start: your free key
Notebooks 1–4 needed nothing. This one needs a **free Groq API key**.

1. Go to **console.groq.com** and sign up (free, no card)
2. Create an API key and copy it
3. In Colab, click the **🔑 key icon** in the left sidebar
4. Add a secret named exactly **`GROQ_API_KEY`**, paste the key in
5. Switch **Notebook access** on

> **Never paste your key into a box in this notebook.** Anyone you share the
> notebook with would get your key — and your bill. The Secrets panel keeps it hidden.

In [ ]:
#@title ▶ Step 1 — Check my key is set up { display-mode: "form" }
#@markdown Nothing to fill in. Just press play.

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

import json as _json
BASE_URL = "https://api.groq.com/openai/v1/chat/completions"
MODELS_URL = "https://api.groq.com/openai/v1/models"
PREFERRED = ["llama-3.3-70b-versatile", "llama-3.1-8b-instant",
             "llama3-70b-8192", "llama3-8b-8192", "mixtral-8x7b-32768", "gemma2-9b-it",
             "openai/gpt-oss-120b", "openai/gpt-oss-20b"]
SKIP = ("whisper", "tts", "guard", "orpheus", "embed")
def usable_models(k):
    import requests
    try:
        r = requests.get(MODELS_URL, headers={"Authorization": "Bearer " + k}, timeout=30)
        if r.status_code != 200: return []
        ids = [m["id"] for m in r.json().get("data", [])]
    except Exception:
        return []
    return [i for i in ids if not any(s in i.lower() for s in SKIP)]
def choose_model(k):
    if globals().get("_MODEL"): return globals()["_MODEL"]
    avail = usable_models(k)
    pick = next((m for m in PREFERRED if m in avail), None) or (avail[0] if avail else None)
    if pick: globals()["_MODEL"] = pick
    return pick
def get_key():
    try:
        from google.colab import userdata
        k = userdata.get("GROQ_API_KEY")
        if k and k.strip(): return k.strip()
    except Exception:
        pass
    import os
    k = os.environ.get("GROQ_API_KEY")
    return k.strip() if k else None
def key_missing():
    problem("I could not find your GROQ_API_KEY.")
    note("One-time setup, takes two minutes:")
    note("   1. Get a free key at console.groq.com (no card needed)")
    note("   2. In Colab, click the key icon in the left sidebar")
    note("   3. Add a secret named exactly  GROQ_API_KEY  and paste the key in")
    note("   4. Turn ON 'Notebook access' for it")
    note("   5. Press play on this step again")
    note("Never paste the key into a box in this notebook - anyone you share it with would see it.")
def mask(k):
    return k[:4] + "..." + k[-4:] if k and len(k) > 10 else "????"
def ask(messages, temperature=0.0, max_tokens=600, _retry=False):
    import requests
    k = get_key()
    if not k: key_missing(); return None
    if not choose_model(k):
        problem("Your key works, but I could not find a model it can use.")
        note("Check console.groq.com to see which models your account has access to.")
        return None
    try:
        r = requests.post(BASE_URL,
                          headers={"Authorization": "Bearer " + k, "Content-Type": "application/json"},
                          json={"model": choose_model(k), "messages": messages,
                                "temperature": temperature, "max_tokens": max_tokens},
                          timeout=60)
    except Exception as e:
        problem("Could not reach the server: " + str(e)[:90]); return None
    if r.status_code == 401:
        problem("The server rejected your key (401)."); note("Check the key in Colab Secrets is correct and still active."); return None
    if r.status_code == 429:
        problem("Too many requests (429). Wait a moment and press play again."); return None
    if r.status_code != 200:
        problem("The server replied with an error " + str(r.status_code) + "."); note(r.text[:160]); return None
    data = r.json()
    msg = data["choices"][0]["message"]
    if not (msg.get("content") or "").strip():
        if not _retry:
            # reasoning models spend their budget thinking before answering
            return ask(messages, temperature=temperature, max_tokens=max(1200, max_tokens * 4), _retry=True)
        think = (msg.get("reasoning") or "").strip()
        if think:
            note("This model thinks before answering and ran out of room for the final answer.")
            msg["content"] = think
        else:
            problem("The model sent back an empty reply."); note("Try a shorter question."); return None
    return data

k = get_key()
if not k:
    key_missing()
else:
    success("Found your key: " + mask(k))
    m = choose_model(k)
    if m:
        success("Your twin's brain: " + m)
        others = [x for x in usable_models(k) if x != m]
        if others: note("Other models your key can use: " + ", ".join(others)[:150])
    note("Only the first and last few characters are shown - never the whole key.")
    print("")
    print("Saying hello to the model to check it really works...")
    print("")
    out = ask([{"role": "user", "content": "Say hello in one short sentence."}], temperature=0.0, max_tokens=400)
    if out:
        success("It replied:")
        print("   " + out["choices"][0]["message"]["content"].strip())
        print("")
        print("You are connected to a model running on someone else's computer,")
        print("somewhere in the world. That round trip took about a second.")

---
## Part 1: The five pieces of every API call

Every request to every AI service is built from the same five parts. Run the step
below to see yours laid out.

In [ ]:
#@title ▶ Step 2 — Show me the five pieces of the call { display-mode: "form" }
#@markdown Change the message and watch which piece it lands in.
my_message = "Explain what an API is, in one sentence a 15-year-old would understand." #@param {type:"string"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

import json as _json
BASE_URL = "https://api.groq.com/openai/v1/chat/completions"
MODELS_URL = "https://api.groq.com/openai/v1/models"
PREFERRED = ["llama-3.3-70b-versatile", "llama-3.1-8b-instant",
             "llama3-70b-8192", "llama3-8b-8192", "mixtral-8x7b-32768", "gemma2-9b-it",
             "openai/gpt-oss-120b", "openai/gpt-oss-20b"]
SKIP = ("whisper", "tts", "guard", "orpheus", "embed")
def usable_models(k):
    import requests
    try:
        r = requests.get(MODELS_URL, headers={"Authorization": "Bearer " + k}, timeout=30)
        if r.status_code != 200: return []
        ids = [m["id"] for m in r.json().get("data", [])]
    except Exception:
        return []
    return [i for i in ids if not any(s in i.lower() for s in SKIP)]
def choose_model(k):
    if globals().get("_MODEL"): return globals()["_MODEL"]
    avail = usable_models(k)
    pick = next((m for m in PREFERRED if m in avail), None) or (avail[0] if avail else None)
    if pick: globals()["_MODEL"] = pick
    return pick
def get_key():
    try:
        from google.colab import userdata
        k = userdata.get("GROQ_API_KEY")
        if k and k.strip(): return k.strip()
    except Exception:
        pass
    import os
    k = os.environ.get("GROQ_API_KEY")
    return k.strip() if k else None
def key_missing():
    problem("I could not find your GROQ_API_KEY.")
    note("One-time setup, takes two minutes:")
    note("   1. Get a free key at console.groq.com (no card needed)")
    note("   2. In Colab, click the key icon in the left sidebar")
    note("   3. Add a secret named exactly  GROQ_API_KEY  and paste the key in")
    note("   4. Turn ON 'Notebook access' for it")
    note("   5. Press play on this step again")
    note("Never paste the key into a box in this notebook - anyone you share it with would see it.")
def mask(k):
    return k[:4] + "..." + k[-4:] if k and len(k) > 10 else "????"
def ask(messages, temperature=0.0, max_tokens=600, _retry=False):
    import requests
    k = get_key()
    if not k: key_missing(); return None
    if not choose_model(k):
        problem("Your key works, but I could not find a model it can use.")
        note("Check console.groq.com to see which models your account has access to.")
        return None
    try:
        r = requests.post(BASE_URL,
                          headers={"Authorization": "Bearer " + k, "Content-Type": "application/json"},
                          json={"model": choose_model(k), "messages": messages,
                                "temperature": temperature, "max_tokens": max_tokens},
                          timeout=60)
    except Exception as e:
        problem("Could not reach the server: " + str(e)[:90]); return None
    if r.status_code == 401:
        problem("The server rejected your key (401)."); note("Check the key in Colab Secrets is correct and still active."); return None
    if r.status_code == 429:
        problem("Too many requests (429). Wait a moment and press play again."); return None
    if r.status_code != 200:
        problem("The server replied with an error " + str(r.status_code) + "."); note(r.text[:160]); return None
    data = r.json()
    msg = data["choices"][0]["message"]
    if not (msg.get("content") or "").strip():
        if not _retry:
            # reasoning models spend their budget thinking before answering
            return ask(messages, temperature=temperature, max_tokens=max(1200, max_tokens * 4), _retry=True)
        think = (msg.get("reasoning") or "").strip()
        if think:
            note("This model thinks before answering and ran out of room for the final answer.")
            msg["content"] = think
        else:
            problem("The model sent back an empty reply."); note("Try a shorter question."); return None
    return data

k = get_key()
print("1. BASE URL - the address you send to")
print("     " + BASE_URL)
print("")
print("2. YOUR KEY - proves who you are")
print("     " + (mask(k) if k else "not set yet"))
print("")
print("3. HEADERS - the envelope, saying who you are and that you are sending JSON")
print('     Authorization: Bearer ' + (mask(k) if k else "YOUR_KEY"))
print('     Content-Type: application/json')
print("")
print("4. MODEL - which brain you want to talk to")
print("     " + (choose_model(k) if k else "chosen automatically from what your key can use"))
print("")
print("5. MESSAGES - the actual conversation")
print('     [{"role": "user", "content": "' + my_message.strip()[:52] + '..."}]')
print("")
print("That is the whole thing. Swap the base URL and the model name and the")
print("same five pieces talk to OpenAI, Gemini, or anyone else.")
print("")
if k:
    print("Sending it for real...")
    print("")
    out = ask([{"role": "user", "content": my_message.strip()}])
    if out:
        print("THE REPLY:")
        print("")
        for line in out["choices"][0]["message"]["content"].strip().split(chr(10)):
            print("   " + line)
        u = out.get("usage", {})
        print("")
        note("Tokens used - sent: " + str(u.get("prompt_tokens", "?")) +
             ", received: " + str(u.get("completion_tokens", "?")) +
             ", total: " + str(u.get("total_tokens", "?")) + ". This is what you are billed on.")
else:
    key_missing()

---
## Part 2: Giving it a personality

The `messages` list has roles. **`system`** sets the standing rules — exactly the brief
you wrote in notebook 4 — and **`user`** is the person talking.

In [ ]:
#@title ▶ Step 3 — Give the model a personality and test it { display-mode: "form" }
#@markdown The standing rules (this is your system instruction):
system_instruction = "You are a digital twin of a 17-year-old student from Accra who loves football. Answer briefly, in a friendly way. Never claim to be a real human." #@param {type:"string"}
#@markdown What the person says:
user_message = "Who are you, and what do you do on Saturdays?" #@param {type:"string"}
#@markdown How adventurous should it be? 0 = predictable, 1 = creative
temperature = 0.3 #@param {type:"slider", min:0, max:1, step:0.1}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

import json as _json
BASE_URL = "https://api.groq.com/openai/v1/chat/completions"
MODELS_URL = "https://api.groq.com/openai/v1/models"
PREFERRED = ["llama-3.3-70b-versatile", "llama-3.1-8b-instant",
             "llama3-70b-8192", "llama3-8b-8192", "mixtral-8x7b-32768", "gemma2-9b-it",
             "openai/gpt-oss-120b", "openai/gpt-oss-20b"]
SKIP = ("whisper", "tts", "guard", "orpheus", "embed")
def usable_models(k):
    import requests
    try:
        r = requests.get(MODELS_URL, headers={"Authorization": "Bearer " + k}, timeout=30)
        if r.status_code != 200: return []
        ids = [m["id"] for m in r.json().get("data", [])]
    except Exception:
        return []
    return [i for i in ids if not any(s in i.lower() for s in SKIP)]
def choose_model(k):
    if globals().get("_MODEL"): return globals()["_MODEL"]
    avail = usable_models(k)
    pick = next((m for m in PREFERRED if m in avail), None) or (avail[0] if avail else None)
    if pick: globals()["_MODEL"] = pick
    return pick
def get_key():
    try:
        from google.colab import userdata
        k = userdata.get("GROQ_API_KEY")
        if k and k.strip(): return k.strip()
    except Exception:
        pass
    import os
    k = os.environ.get("GROQ_API_KEY")
    return k.strip() if k else None
def key_missing():
    problem("I could not find your GROQ_API_KEY.")
    note("One-time setup, takes two minutes:")
    note("   1. Get a free key at console.groq.com (no card needed)")
    note("   2. In Colab, click the key icon in the left sidebar")
    note("   3. Add a secret named exactly  GROQ_API_KEY  and paste the key in")
    note("   4. Turn ON 'Notebook access' for it")
    note("   5. Press play on this step again")
    note("Never paste the key into a box in this notebook - anyone you share it with would see it.")
def mask(k):
    return k[:4] + "..." + k[-4:] if k and len(k) > 10 else "????"
def ask(messages, temperature=0.0, max_tokens=600, _retry=False):
    import requests
    k = get_key()
    if not k: key_missing(); return None
    if not choose_model(k):
        problem("Your key works, but I could not find a model it can use.")
        note("Check console.groq.com to see which models your account has access to.")
        return None
    try:
        r = requests.post(BASE_URL,
                          headers={"Authorization": "Bearer " + k, "Content-Type": "application/json"},
                          json={"model": choose_model(k), "messages": messages,
                                "temperature": temperature, "max_tokens": max_tokens},
                          timeout=60)
    except Exception as e:
        problem("Could not reach the server: " + str(e)[:90]); return None
    if r.status_code == 401:
        problem("The server rejected your key (401)."); note("Check the key in Colab Secrets is correct and still active."); return None
    if r.status_code == 429:
        problem("Too many requests (429). Wait a moment and press play again."); return None
    if r.status_code != 200:
        problem("The server replied with an error " + str(r.status_code) + "."); note(r.text[:160]); return None
    data = r.json()
    msg = data["choices"][0]["message"]
    if not (msg.get("content") or "").strip():
        if not _retry:
            # reasoning models spend their budget thinking before answering
            return ask(messages, temperature=temperature, max_tokens=max(1200, max_tokens * 4), _retry=True)
        think = (msg.get("reasoning") or "").strip()
        if think:
            note("This model thinks before answering and ran out of room for the final answer.")
            msg["content"] = think
        else:
            problem("The model sent back an empty reply."); note("Try a shorter question."); return None
    return data

if not system_instruction.strip() or not user_message.strip():
    problem("Both boxes need filling in.")
else:
    msgs = [{"role": "system", "content": system_instruction.strip()},
            {"role": "user", "content": user_message.strip()}]
    print("WHAT GETS SENT:")
    print("")
    print('   system: "' + system_instruction.strip()[:64] + '..."')
    print('   user:   "' + user_message.strip()[:64] + '"')
    print("")
    out = ask(msgs, temperature=float(temperature))
    if out:
        print("THE REPLY:")
        print("")
        for line in out["choices"][0]["message"]["content"].strip().split(chr(10)):
            print("   " + line)
        print("")
        print("So what? You did not retrain anything. One sentence of instruction")
        print("changed who the model pretends to be.")
        note("Try editing the system box - make it a farmer, a teacher, a nurse - and run it again.")
        note("Then try the temperature slider: run the same question twice at 0, then twice at 1.")

---
## Part 3: Memory

The server forgets you the instant it replies. If you want a conversation, **you**
have to send the whole history back every time. Run this step several times and watch
the conversation build.

In [ ]:
#@title ▶ Step 4 — Have a real conversation (run me again and again) { display-mode: "form" }
#@markdown Say something. Press play. Then change this and press play again.
what_i_say = "My name is Ama and I play football on Saturdays." #@param {type:"string"}
#@markdown Tick to wipe the conversation and start fresh:
start_over = False #@param {type:"boolean"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

import json as _json
BASE_URL = "https://api.groq.com/openai/v1/chat/completions"
MODELS_URL = "https://api.groq.com/openai/v1/models"
PREFERRED = ["llama-3.3-70b-versatile", "llama-3.1-8b-instant",
             "llama3-70b-8192", "llama3-8b-8192", "mixtral-8x7b-32768", "gemma2-9b-it",
             "openai/gpt-oss-120b", "openai/gpt-oss-20b"]
SKIP = ("whisper", "tts", "guard", "orpheus", "embed")
def usable_models(k):
    import requests
    try:
        r = requests.get(MODELS_URL, headers={"Authorization": "Bearer " + k}, timeout=30)
        if r.status_code != 200: return []
        ids = [m["id"] for m in r.json().get("data", [])]
    except Exception:
        return []
    return [i for i in ids if not any(s in i.lower() for s in SKIP)]
def choose_model(k):
    if globals().get("_MODEL"): return globals()["_MODEL"]
    avail = usable_models(k)
    pick = next((m for m in PREFERRED if m in avail), None) or (avail[0] if avail else None)
    if pick: globals()["_MODEL"] = pick
    return pick
def get_key():
    try:
        from google.colab import userdata
        k = userdata.get("GROQ_API_KEY")
        if k and k.strip(): return k.strip()
    except Exception:
        pass
    import os
    k = os.environ.get("GROQ_API_KEY")
    return k.strip() if k else None
def key_missing():
    problem("I could not find your GROQ_API_KEY.")
    note("One-time setup, takes two minutes:")
    note("   1. Get a free key at console.groq.com (no card needed)")
    note("   2. In Colab, click the key icon in the left sidebar")
    note("   3. Add a secret named exactly  GROQ_API_KEY  and paste the key in")
    note("   4. Turn ON 'Notebook access' for it")
    note("   5. Press play on this step again")
    note("Never paste the key into a box in this notebook - anyone you share it with would see it.")
def mask(k):
    return k[:4] + "..." + k[-4:] if k and len(k) > 10 else "????"
def ask(messages, temperature=0.0, max_tokens=600, _retry=False):
    import requests
    k = get_key()
    if not k: key_missing(); return None
    if not choose_model(k):
        problem("Your key works, but I could not find a model it can use.")
        note("Check console.groq.com to see which models your account has access to.")
        return None
    try:
        r = requests.post(BASE_URL,
                          headers={"Authorization": "Bearer " + k, "Content-Type": "application/json"},
                          json={"model": choose_model(k), "messages": messages,
                                "temperature": temperature, "max_tokens": max_tokens},
                          timeout=60)
    except Exception as e:
        problem("Could not reach the server: " + str(e)[:90]); return None
    if r.status_code == 401:
        problem("The server rejected your key (401)."); note("Check the key in Colab Secrets is correct and still active."); return None
    if r.status_code == 429:
        problem("Too many requests (429). Wait a moment and press play again."); return None
    if r.status_code != 200:
        problem("The server replied with an error " + str(r.status_code) + "."); note(r.text[:160]); return None
    data = r.json()
    msg = data["choices"][0]["message"]
    if not (msg.get("content") or "").strip():
        if not _retry:
            # reasoning models spend their budget thinking before answering
            return ask(messages, temperature=temperature, max_tokens=max(1200, max_tokens * 4), _retry=True)
        think = (msg.get("reasoning") or "").strip()
        if think:
            note("This model thinks before answering and ran out of room for the final answer.")
            msg["content"] = think
        else:
            problem("The model sent back an empty reply."); note("Try a shorter question."); return None
    return data

if start_over or "_CHAT" not in globals():
    globals()["_CHAT"] = [{"role": "system", "content":
        globals().get("_SYSTEM_TXT", "You are a helpful digital twin. Keep answers short and friendly.")}]
    note("Starting a fresh conversation.")
    print("")
chat = globals()["_CHAT"]
if not what_i_say.strip():
    problem("Type something to say first.")
else:
    chat.append({"role": "user", "content": what_i_say.strip()})
    out = ask(chat, temperature=0.4)
    if out is None:
        chat.pop()
    else:
        reply = out["choices"][0]["message"]["content"].strip()
        chat.append({"role": "assistant", "content": reply})
        turns = sum(1 for m in chat if m["role"] == "user")
        print("THE CONVERSATION SO FAR (" + str(turns) + " turn" + ("s" if turns != 1 else "") + "):")
        print("")
        for m in chat:
            if m["role"] == "system": continue
            who = "You " if m["role"] == "user" else "Twin"
            for i, line in enumerate(m["content"].split(chr(10))):
                print(("   " + who + ": " if i == 0 else "         ") + line)
        print("")
        u = out.get("usage", {})
        note("This turn sent " + str(u.get("prompt_tokens", "?")) + " tokens - the WHOLE conversation, " +
             "not just your last line.")
        print("")
        if turns == 1:
            print("Now change the box above to 'What is my name?' and press play again.")
            print("It will remember - because the history goes with every request.")
        else:
            print("So what? The model has no memory. You are re-sending the entire chat")
            print("every single time, and that is the whole illusion.")
            note("Notice the token count climbing. That is why long chats cost more,")
            note("and why Week 2 is about managing what you send.")

---
## Summary & Key Takeaways

1. **Five pieces** — base URL, key, headers, model, messages. Every provider, same shape.
2. **Keys live in Secrets**, never in a cell. A leaked key is someone else's bill.
3. **Roles** — `system` sets the rules, `user` is the person. One sentence changes the personality.
4. **Temperature** — low for predictable, high for creative.
5. **No memory** — you re-send the whole conversation every turn, and pay for it every turn.

> Your digital twin is now a real, live thing anyone could talk to.

### Next up
Week 2: handling messy text, staying inside the token limit, and giving your twin
real knowledge to draw on.